In [1]:
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import logit, expit
from sklearn import metrics

import pandas as pd


import random
import yaml

#from ..dataset.abstract_dataset import DeepfakeAbstractBaseDataset

from datasets.abstract_dataset import DeepfakeAbstractBaseDataset
from detectors import DETECTOR


#from metrics.utils import get_test_metrics

In [2]:
def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)
    
def init_seed(config):
    if config['manualSeed'] is None:
        config['manualSeed'] = random.randint(1, 10000)
    random.seed(config['manualSeed'])
    torch.manual_seed(config['manualSeed'])
    if config['cuda']:
        torch.cuda.manual_seed_all(config['manualSeed'])

In [3]:
# Adapted from test.py

def prepare_testing_data(config):
    def get_test_data_loader(config, test_name):
        # update the config dictionary with the specific testing dataset
        config = config.copy()  # create a copy of config to avoid altering the original one
        config['test_dataset'] = test_name  # specify the current test dataset
        test_set = DeepfakeAbstractBaseDataset(
                config=config,
                mode='test', 
            )
        
        test_data_loader = \
            torch.utils.data.DataLoader(
                dataset=test_set, 
                batch_size=config['test_batchSize'],
                shuffle=False, 
                num_workers=int(config['workers']),
                collate_fn=test_set.collate_fn,
                drop_last=False
            )
        return test_data_loader

    test_data_loaders = {}
    for one_test_name in config['test_dataset']:
        test_data_loaders[one_test_name] = get_test_data_loader(config, one_test_name)
    return test_data_loaders

In [4]:
@torch.no_grad()
def inference(model, data_dict):
    predictions = model(data_dict, inference=True)
    return predictions

def test_one_dataset(model, data_loader, device):
    prediction_lists = []
    #feature_lists = []
    label_lists = []
    for i, data_dict in tqdm(enumerate(data_loader), total=len(data_loader)):
        # get data
        data, label, mask, landmark = \
        data_dict['image'], data_dict['label'], data_dict['mask'], data_dict['landmark']
        label = torch.where(data_dict['label'] != 0, 1, 0)
        # move data to GPU
        data_dict['image'], data_dict['label'] = data.to(device), label.to(device)
        if mask is not None:
            data_dict['mask'] = mask.to(device)
        if landmark is not None:
            data_dict['landmark'] = landmark.to(device)

        # model forward without considering gradient computation
        predictions = inference(model, data_dict)
        label_lists += list(data_dict['label'].cpu().detach().numpy())

        # if type(model).__name__ == "UCFDetector":
        #     prediction_lists += predictions['prob']
        # else:
        #     
        prediction_lists += list(predictions['prob'].cpu().detach().numpy())
        #feature_lists += list(predictions['feat'].cpu().detach().numpy())
    
    return np.array(prediction_lists), np.array(label_lists)#np.array(feature_lists)

In [5]:
def test_epoch(model, test_data_loaders, device):

    # set model to eval mode
    model.eval()

    # define test recorder
    metrics_all_datasets = {}
    preds_labels_paths = {}


    # testing for all test data
    keys = test_data_loaders.keys()
    for key in keys:
        data_dict = test_data_loaders[key].dataset.data_dict
        # compute loss for each dataset
        predictions_nps, label_nps = test_one_dataset(model, test_data_loaders[key], device)

        return (predictions_nps, label_nps, data_dict['image'])
        
        # compute metric for each dataset
        metric_one_dataset, preds_labels_paths_one_dataset = get_test_metrics(y_pred=predictions_nps, y_true=label_nps,
                                              img_names=data_dict['image'])
        metrics_all_datasets[key] = metric_one_dataset
        preds_labels_paths[key] = preds_labels_paths_one_dataset
        
        # info for each dataset
        tqdm.write(f"dataset: {key}")
        for k, v in metric_one_dataset.items():

            if k == "pred" or k == "label" or k == "paths":
                continue

            tqdm.write(f"{k}: {v}")

    #return metrics_all_datasets, preds_labels_paths

In [6]:
def init(config_path, weights_path, test_dataset, config_test):
        config = load_config(config_path)
        test_config = load_config(config_test)

        config.update(test_config)
        config["test_dataset"] = test_dataset
        config["weights_path"] = weights_path

        if config['cudnn']:
                cudnn.benchmark = True

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        init_seed(config)


        test_data_loaders = prepare_testing_data(config)

        model_class = DETECTOR[config['model_name']]
        model = model_class(config).to(device)

        ckpt = torch.load(config["weights_path"], map_location=device)

        if 'state_dict' in ckpt:
                ckpt = ckpt['state_dict']

        new_weights = {}

        for key, value in ckpt.items():
                new_key = key.replace('module.', '')
                if 'base_model.' in new_key:
                        new_key = new_key.replace('base_model.', 'backbone.')
                if 'classifier.' in new_key:
                        new_key = new_key.replace('classifier.', 'head.')
                if 'HRNet_layer.' in new_key:
                        new_key = new_key.replace('HRNet_layer.', 'backbone.')
                new_weights[new_key] = value

        if type(model).__name__ == "EffortDetector":
                model.load_state_dict(new_weights, strict=False)
        else:
                model.load_state_dict(new_weights, strict=True)
        print('===> Load checkpoint done!')

        return test_data_loaders, model, device


In [7]:
def get_prediction_df(predictions_nps, labels_nps, images):

    videos_real = set([p.split('/')[9] for p in images if p.split('/')[7] == 'real'])
    videos_fake = set([p.split('/')[9] for p in images if p.split('/')[7] == 'fake'])

    all_videos = videos_fake.union(videos_real)

    assert(len(videos_real) == 42)
    assert(len(videos_fake) == 42)

    df_by_frame = pd.DataFrame(images, columns=["video_path"])
    df_by_frame["video_name"] = [p.split('/')[9] for p in images]
    df_by_frame["pred"] = predictions_nps
    df_by_frame["label"] = labels_nps

    df_video_preds = pd.DataFrame(all_videos, columns=["video_name"])
    df_video_preds["label"] = df_video_preds.apply(lambda row: 1 if row["video_name"] in videos_fake else 0, axis=1)


    for video_name in all_videos:
        video_frames = df_by_frame[df_by_frame["video_name"] == video_name]

        #preds_sum = video_frames["preds"].sum()

        preds_clipped = np.clip(video_frames["pred"], 1e-7, 1 - 1e-7)
        preds_logit = logit(preds_clipped)
        avg_logit = np.mean(preds_logit)
        prediction_class_log = 0 if expit(avg_logit) < 0.5 else 1

        preds_mean = video_frames['pred'].mean()
        prediction_class_mean = 0 if preds_mean < 0.5 else 1

        if prediction_class_log != prediction_class_mean:
            print("mismatch", f"pred log {expit(avg_logit)}, pred normal {preds_mean}, class log {prediction_class_log}, class normal {prediction_class_mean}, vid {video_name}")

        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_log"] = prediction_class_log
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_mean"] = prediction_class_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_log"] = expit(avg_logit)
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_mean"] = preds_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "max_pred"] = video_frames['pred'].max()

    return df_video_preds

In [ ]:
def get_prediction_df_video_level(predictions_nps, labels_nps, images):

    preds_by_video = {}
    all_videos = []
    label_by_video = {}

    for index, image in enumerate(images):
        video_name = image[0].split('/')[9]
        
        if video_name not in label_by_video.keys():
            label_by_video[video_name] = labels_nps[index]
        all_videos.append(video_name)

        if video_name in preds_by_video.keys():
            preds_by_video[video_name] = [*preds_by_video[video_name], *predictions_nps[index]]
        else:
            preds_by_video[video_name] = predictions_nps[index]


    all_videos = set(all_videos)

    assert(len(all_videos) == 84)

    df_video_preds = pd.DataFrame(all_videos, columns=["video_name"])

    df_video_preds["label"] = df_video_preds.iloc[:, 0].map(label_by_video)


    for video_name in all_videos:

        preds_clipped = np.clip(preds_by_video[video_name], 1e-7, 1 - 1e-7)
        preds_logit = logit(preds_clipped)
        avg_logit = np.mean(preds_logit)
        prediction_class_log = 0 if expit(avg_logit) < 0.5 else 1

        df_preds = pd.DataFrame(preds_by_video[video_name])
        
        preds_mean = df_preds.mean()

        prediction_class_mean = 0 if preds_mean.squeeze() < 0.5 else 1

        if prediction_class_log != prediction_class_mean:
            print("mismatch", f"pred log {expit(avg_logit)}, pred normal {preds_mean}, class log {prediction_class_log}, class normal {prediction_class_mean}, vid {video_name}")

        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_log"] = prediction_class_log
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_mean"] = prediction_class_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_log"] = expit(avg_logit)
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_mean"] = preds_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "max_pred"] = max(preds_by_video[video_name])


    return df_video_preds

# Entry Point

In [ ]:
exps = pd.read_csv("exps.csv")

CONFIG_TEST = "config/test_config.yaml"
CONFIG_DETECTORS = "config/detector/"
WEIGHTS_BASE = "weights/"

for _, row in exps.iterrows():
    model_name = row['model_name']
    config_detector_path = row['config']
    weights_path = row ['weights']
    test_dataset = row['test_dataset']

    test_data_loaders, model, device = init(CONFIG_DETECTORS + config_detector_path, WEIGHTS_BASE + weights_path, [test_dataset], CONFIG_TEST)

    predictions_nps, label_nps, images = test_epoch(model, test_data_loaders, device)

    #if "i3d" in detector_name or "altfreezing" in detector_name:
    #    detector_results = get_prediction_df_video_level(predictions_nps, label_nps, images)
    #else:
    detector_results = get_prediction_df(predictions_nps, label_nps, images)

    #write_results(detector_results, detector_config_path.split("/")[-1].replace(".yaml", ""), weights[index].split("/")[-1].replace(".pth", ""))